# S05 J2 — Architecture des transformations



## 1. Vue d’ensemble



Bronze → Déduplication
ROW_NUMBER sur clé métier + ingestion_timestamp

Déduplication → Explosion
explode_outer sur les collections

Explosion → Qualité
règles sur clés, quantités, prix, dates

Qualité → Silver
uniquement les lignes valides

Qualité → Quarantaine
lignes invalides + quality_reason

Silver → Gold
agrégations au grain métier

## 2. Contrat Bronze




### Grain
1 ligne = 1 occurrence reçue d’une commande.

### Clés
- Clé métier : `order_id`
- Clé technique d’occurrence : `order_id + ingestion_timestamp`

### Schéma
- `order_id`
- `customer_id`
- `order_date`
- `order_items`
- `ingestion_timestamp`
- `source_file`

### Responsabilité
La couche Bronze conserve les données reçues avec un minimum de transformation, garde la traçabilité de l’ingestion et permet de rejouer les traitements vers Silver.

## 3. Contrat Silver

### Grain
1 ligne = 1 article d’une commande après déduplication et explosion de `order_items`.

### Clé logique
`order_id + item_id`

### Schéma
- `order_id`
- `customer_id`
- `order_date`
- `item_id`
- `quantity`
- `price`
- `ingestion_timestamp`
- `source_file`

### Responsabilité
La couche Silver contient des données :
- dédupliquées ;
- structurées ;
- normalisées ;
- validées par les règles de qualité ;
- exploitables pour les transformations métier.

Les lignes invalides sont redirigées vers la quarantaine.

## 4. Contrat Gold

### Grain
1 ligne = 1 date × 1 client.

### Clé logique
`order_date + customer_id`

### Schéma
- `order_date`
- `customer_id`
- `revenue`
- `quantity_sold`

### Responsabilité
La couche Gold contient des données agrégées et orientées métier, directement exploitables pour la BI et le reporting.

Chaque métrique doit être recalculable à partir de la Silver.



## 5. Jointures et risques

### Risque de perte de lignes

Un `INNER JOIN` supprime les lignes qui ne trouvent aucune correspondance.

Pour préserver toutes les lignes de la table principale, utiliser un `LEFT JOIN`.

Exemple :
- commandes = table principale ;
- référentiel client = table de droite ;
- client inconnu → conservé avec une valeur `UNKNOWN`.

### Risque de multiplication

Si la clé de jointure n’est pas unique dans la table de droite, une ligne de gauche peut produire plusieurs lignes.

Avant une jointure importante :
- vérifier l’unicité de la clé ;
- comparer `COUNT(*)` et `COUNT(DISTINCT key)` ;
- identifier les doublons ;
- dédupliquer la référence si nécessaire.

## 6. explode vs explode_outer

### explode
Transforme chaque élément d’une collection en une ligne.

Risque :
les collections vides ou `NULL` peuvent faire disparaître la ligne source.

### explode_outer
Transforme également chaque élément en une ligne, mais conserve les lignes dont la collection est vide ou `NULL`.

### Choix d’architecture
Utiliser `explode_outer` lorsqu’il est important de :
- préserver la traçabilité ;
- éviter une perte silencieuse de données ;
- appliquer ensuite explicitement une règle de qualité sur les collections vides ou nulles.



## 7. Déduplication

### Clé métier
`order_id`

### Règle
Lorsqu’une même commande est reçue plusieurs fois, conserver la version la plus récente selon `ingestion_timestamp`.

### Implémentation
Utiliser `ROW_NUMBER()` avec une fenêtre :

```sql
ROW_NUMBER() OVER (
  PARTITION BY order_id
  ORDER BY ingestion_timestamp DESC
) AS rn


## 8. Qualité et quarantaine

### Principe

Les lignes invalides ne sont pas supprimées silencieusement.

Chaque échec doit être :
- détecté ;
- expliqué ;
- observable ;
- réconciliable avec les volumes d’entrée.

### Règles de qualité

- `order_id IS NOT NULL`
- `customer_id IS NOT NULL`
- `item_id IS NOT NULL`
- `quantity > 0`
- `price >= 0`

### Routage

- `VALID` → Silver
- `INVALID_*` → Quarantaine

### Exemple de raison

```sql
CASE
  WHEN order_id IS NULL THEN 'INVALID_ORDER_ID'
  WHEN customer_id IS NULL THEN 'INVALID_CUSTOMER_ID'
  WHEN item_id IS NULL THEN 'INVALID_ITEM_ID'
  WHEN quantity <= 0 THEN 'INVALID_QUANTITY'
  WHEN price < 0 THEN 'INVALID_PRICE'
  ELSE 'VALID'
END AS quality_status


## 9. Réconciliation

### Principe

Chaque transformation importante doit être contrôlée avec des volumes ou des clés métier.

Lorsque le grain est identique :

`entrée = sortie valide + quarantaine`

Exemple du lab :

- lignes après `explode_outer` : 14
- lignes Silver : 12
- lignes Quarantaine : 2

Donc :

`14 = 12 + 2`

La réconciliation est correcte.

### Attention au changement de grain

Il ne faut pas comparer naïvement `COUNT(*)` entre Bronze et Silver si le grain change.

Exemple :

- Bronze : 1 ligne = 1 commande
- Silver : 1 ligne = 1 article de commande

Dans ce cas, on réconcilie plutôt avec :
- les clés métier ;
- `COUNT(DISTINCT order_id)` ;
- ou un contrôle de traçabilité par commande.

## 10. Décisions d’architecture

- Bronze conserve les occurrences reçues et la traçabilité.
- La déduplication se fait avant l’explosion des collections.
- `ROW_NUMBER()` permet de garder la version la plus récente.
- `explode_outer` est préféré pour éviter une perte silencieuse des commandes sans article.
- Les contrôles qualité sont centralisés avant le routage.
- Les lignes valides alimentent Silver.
- Les lignes invalides alimentent la Quarantaine avec une raison explicite.
- Silver reste au grain fin.
- Gold agrège Silver selon un grain métier explicite.
- Chaque métrique Gold doit être recalculable depuis Silver.
- Les volumes doivent être réconciliables entre les étapes.